In [1]:
# --- SCRIPT 1: GRAND AGGREGATION - 3B STANDARD (NATIVE) RESOLUTION ---
import os
import json
import csv

# --- CONFIGURATION ---
INPUT_FILES = [
    "/kaggle/input/datasets/mianafzalsaeed/nlp-mini-project-3b-standard-results/3B_ablation_person_A_results.json",
    "/kaggle/input/datasets/mianafzalsaeed/nlp-mini-project-3b-standard-results/3B_ablation_person_B_results.json",
    "/kaggle/input/datasets/mianafzalsaeed/nlp-mini-project-3b-standard-results/3B_ablation_person_C_results.json",
    "/kaggle/input/datasets/mianafzalsaeed/nlp-mini-project-3b-standard-results/3B_ablation_person_D_results.json",
    "/kaggle/input/datasets/mianafzalsaeed/nlp-mini-project-3b-standard-results/3B_ablation_person_A_patch_600-799.json"
]

MASTER_JSON_OUTPUT = "/kaggle/working/MASTER_3B_STANDARD_Results.json"
MASTER_CSV_OUTPUT = "/kaggle/working/MASTER_3B_STANDARD_Metrics.csv"

MVBENCH_TASKS = [
    'action_sequence', 'moving_count', 'action_prediction', 'episodic_reasoning', 
    'action_antonym', 'action_count', 'scene_transition', 'object_shuffle', 
    'object_existence', 'fine_grained_pose', 'unexpected_action', 'moving_direction', 
    'state_change', 'object_interaction', 'character_order', 'action_localization', 
    'counterfactual_inference', 'fine_grained_action', 'moving_attribute', 'egocentric_navigation'
]

def get_task_name(index):
    task_idx = index // 200
    if 0 <= task_idx < len(MVBENCH_TASKS): return MVBENCH_TASKS[task_idx]
    return "unknown_task"

print("Commencing Grand Aggregation for 3B STANDARD RESOLUTION...\n")

master_records = {}
for file_path in INPUT_FILES:
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            try:
                data = json.load(f)
                count = 0
                for record in data:
                    if record["id"] not in master_records: count += 1
                    master_records[record["id"]] = record
                print(f"[+] Successfully loaded {count} unique records from {os.path.basename(file_path)}")
            except json.JSONDecodeError:
                print(f"[!] ERROR: {os.path.basename(file_path)} is corrupted.")
    else:
        print(f"[-] WARNING: {os.path.basename(file_path)} not found. Skipping.")

sorted_records = [master_records[k] for k in sorted(master_records.keys())]

task_metrics = {task: {"correct": 0, "total": 0, "ttft_sum": 0.0, "tps_sum": 0.0} for task in MVBENCH_TASKS}
overall_correct, overall_ttft, overall_tps, peak_vram = 0, 0.0, 0.0, 0.0

for r in sorted_records:
    task_name = get_task_name(r["id"])
    if r["response"].strip().upper() == r["ground_truth_letter"].strip().upper():
        task_metrics[task_name]["correct"] += 1
        overall_correct += 1
        
    task_metrics[task_name]["total"] += 1
    task_metrics[task_name]["ttft_sum"] += r.get("ttft_seconds", 0.0)
    task_metrics[task_name]["tps_sum"] += r.get("tokens_per_second", 0.0)
    overall_ttft += r.get("ttft_seconds", 0.0)
    overall_tps += r.get("tokens_per_second", 0.0)
    if r.get("peak_vram_gb", 0.0) > peak_vram: peak_vram = r.get("peak_vram_gb", 0.0)

total_records = len(sorted_records)

print("\n===========================================================================")
print(f"{'MVBench Sub-Task':<30} | {'Accuracy':<10} | {'Avg TTFT':<12} | {'Avg TPS'}")
print("===========================================================================")

csv_data = []
csv_headers = ["Task", "Accuracy_Percent", "Correct", "Total", "Avg_TTFT_sec", "Avg_TPS"]

for task in MVBENCH_TASKS:
    data = task_metrics[task]
    if data["total"] > 0:
        acc, avg_ttft, avg_tps = (data["correct"] / data["total"]) * 100, data["ttft_sum"] / data["total"], data["tps_sum"] / data["total"]
        print(f"{task:<30} | {acc:>5.2f}%    | {avg_ttft:>6.2f} sec   | {avg_tps:>5.2f}")
        csv_data.append([task, round(acc, 2), data["correct"], data["total"], round(avg_ttft, 2), round(avg_tps, 2)])
    else:
        print(f"{task:<30} | NO DATA")

if total_records > 0:
    final_acc, final_avg_ttft, final_avg_tps = (overall_correct / total_records) * 100, overall_ttft / total_records, overall_tps / total_records
    print("===========================================================================")
    print(f"{'OVERALL ACCURACY':<30} | {final_acc:>5.2f}%    ({overall_correct}/{total_records})")
    print("===========================================================================\n")
    print("=============================================")
    print("       HARDWARE PERFORMANCE SUMMARY       ")
    print("=============================================")
    print(f"Average Time-to-First-Token:     {final_avg_ttft:.2f} seconds")
    print(f"Average Tokens-per-Second:       {final_avg_tps:.2f} tokens/s")
    print(f"Absolute Peak VRAM Usage:        {peak_vram:.2f} GB")
    print("=============================================\n")

with open(MASTER_JSON_OUTPUT, "w") as f: json.dump(sorted_records, f, indent=4)
with open(MASTER_CSV_OUTPUT, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(csv_headers)
    writer.writerows(csv_data)
    if total_records > 0:
        writer.writerow([])
        writer.writerow(["OVERALL", round(final_acc, 2), overall_correct, total_records, round(final_avg_ttft, 2), round(final_avg_tps, 2)])
        writer.writerow(["PEAK_VRAM_GB", peak_vram, "", "", "", ""])

print(f"Aggregation Complete! Total Records: {total_records} / 4000")

Commencing Grand Aggregation for 3B STANDARD RESOLUTION...

[+] Successfully loaded 800 unique records from 3B_ablation_person_A_results.json
[+] Successfully loaded 800 unique records from 3B_ablation_person_B_results.json
[+] Successfully loaded 1000 unique records from 3B_ablation_person_C_results.json
[+] Successfully loaded 1000 unique records from 3B_ablation_person_D_results.json
[+] Successfully loaded 200 unique records from 3B_ablation_person_A_patch_600-799.json

MVBench Sub-Task               | Accuracy   | Avg TTFT     | Avg TPS
action_sequence                | 51.50%    |   0.30 sec   | 252.11
moving_count                   | 52.00%    |   0.22 sec   | 36.65
action_prediction              | 43.00%    |   0.26 sec   | 48.74
episodic_reasoning             | 54.00%    |   0.43 sec   | 15.16
action_antonym                 | 73.50%    |   0.26 sec   | 673.81
action_count                   | 45.50%    |   0.26 sec   | 40.47
scene_transition               | 88.50%    |   0.33 se